In [ ]:
#@markdown ### **Installing pip packages**
#@markdown - Diffusion Model: [PyTorch](https://pytorch.org) & [HuggingFace diffusers](https://huggingface.co/docs/diffusers/index)
#@markdown - Dataset Loading: [Zarr](https://zarr.readthedocs.io/en/stable/) & numcodecs
#@markdown - Push-T Env: gym, pygame, pymunk & shapely
!python --version
!pip3 install torch torchvision \
  --extra-index-url https://download.pytorch.org/whl/cu118 --force-reinstall

!pip3 install "numpy<2" scikit-image==0.19.3 --force-reinstall
!pip3 install  diffusers==0.18.2 scikit-video==1.1.11 zarr==2.12.0 numcodecs==0.10.2 \
pygame==2.1.2 pymunk==6.2.1 gym==0.26.2 shapely==1.8.4 

In [3]:
# Local imports
from PushTEnv import PushTEnv
from PushTDataset import PushTStateDataset
from diffusion_model import create_diffusion_model
from training import train_diffusion_model
from inference import run_inference_diffusion_model

In [4]:
#@markdown ### **Imports**
# diffusion policy import
import os
import torch

# from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np

from skvideo.io import vwrite
# from IPython.display import Video
import gdown


In [5]:
from huggingface_hub.utils import IGNORE_GIT_FOLDER_PATTERNS
#@markdown ### **Env Demo**
#@markdown Standard Gym Env (0.21.0 API)

# 0. create env object
env = PushTEnv()

# 1. seed env for initial state.
# Seed 0-200 are used for the demonstration dataset.
env.seed(1000)

# 2. must reset before use
obs, IGNORE_GIT_FOLDER_PATTERNS = env.reset()

# 3. 2D positional action space [0,512]
action = env.action_space.sample()

# 4. Standard gym step method
obs, reward, terminated, truncated, info = env.step(action)

# prints and explains each dimension of the observation and action vectors
with np.printoptions(precision=4, suppress=True, threshold=5):
    print("Obs: ", repr(obs))
    print("Obs:        [agent_x,  agent_y,  block_x,  block_y,    block_angle]")
    print("Action: ", repr(action))
    print("Action:   [target_agent_x, target_agent_y]")

Obs:  array([172.1565, 171.0533, 292.    , 351.    ,   2.9196])
Obs:        [agent_x,  agent_y,  block_x,  block_y,    block_angle]
Action:  array([255.9037, 290.2867])
Action:   [target_agent_x, target_agent_y]


In [6]:
#@markdown ### **Dataset Demo**

# download demonstration data from Google Drive
dataset_path = "pusht_cchi_v7_replay.zarr.zip"
if not os.path.isfile(dataset_path):
    id = "1KY1InLurpMvJDRb14L9NlXT_fEsCvVUq&confirm=t"
    gdown.download(id=id, output=dataset_path, quiet=False)

# parameters
pred_horizon = 16
obs_horizon = 2
action_horizon = 8
#|o|o|                             observations: 2
#| |a|a|a|a|a|a|a|a|               actions executed: 8
#|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16

# create dataset from file
dataset = PushTStateDataset(
    dataset_path=dataset_path,
    pred_horizon=pred_horizon,
    obs_horizon=obs_horizon,
    action_horizon=action_horizon
)
# save training data statistics (min, max) for each dim
stats = dataset.stats

# create dataloader
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=256,
    num_workers=1,
    shuffle=True,
    # accelerate cpu-gpu transfer
    pin_memory=True,
    # don't kill worker process afte each epoch
    persistent_workers=True
)

# visualize data in batch
batch = next(iter(dataloader))
print("batch['obs'].shape:", batch['obs'].shape)
print("batch['action'].shape", batch['action'].shape)

batch['obs'].shape: torch.Size([256, 2, 5])
batch['action'].shape torch.Size([256, 16, 2])


In [7]:
## Diffusion Model

# observation and action dimensions corrsponding to
# the output of PushTEnv
obs_dim = 5
action_dim = 2

# for this demo, we use DDPMScheduler with 100 diffusion iterations
num_diffusion_iters = 100

noise_pred_net, noise_scheduler, device = create_diffusion_model(obs_horizon, pred_horizon, obs_dim, action_dim, num_diffusion_iters)

number of parameters: 6.535322e+07


In [8]:
# Training takes about an hour. If you don't want to wait, skip to the next cell
# to load pre-trained weights
num_epochs = 100
train_diffusion_model(noise_pred_net, noise_scheduler, dataloader, device, 
                          num_epochs, obs_horizon)

Epoch:   8%|▊         | 8/100 [01:21<15:36, 10.18s/it, loss=0.05]  


KeyboardInterrupt: 

In [ ]:
#@markdown ### **Loading Pretrained Checkpoint**
#@markdown Set `load_pretrained = True` to load pretrained weights.

load_pretrained = False
if load_pretrained:
  ckpt_path = "pusht_state_100ep.ckpt"
  if not os.path.isfile(ckpt_path):
      id = "1mHDr_DEZSdiGo9yecL50BBQYzR8Fjhl_&confirm=t"
      gdown.download(id=id, output=ckpt_path, quiet=False)

  state_dict = torch.load(ckpt_path, map_location='cuda')
  ema_noise_pred_net = noise_pred_net
  ema_noise_pred_net.load_state_dict(state_dict)
  print('Pretrained weights loaded.')
else:
  print("Skipped pretrained weight loading.")

Downloading...
From: https://drive.google.com/uc?id=1mHDr_DEZSdiGo9yecL50BBQYzR8Fjhl_&confirm=t
To: /home/mya/Desktop/github/diffusion_exploration/pusht_state_100ep.ckpt
100%|██████████| 261M/261M [00:05<00:00, 50.7MB/s] 


Pretrained weights loaded.


In [10]:

# limit enviornment interaction to 200 steps before termination
max_steps = 200
env = PushTEnv()
# use a seed >200 to avoid initial states seen in the training dataset
env.seed(100000)

rewards, imgs = run_inference_diffusion_model(ema_noise_pred_net, noise_scheduler, env, stats, device, 
        obs_horizon, pred_horizon, action_horizon, action_dim, max_steps, num_diffusion_iters)

# Print out the maximum target coverage
print('Score: ', max(rewards))

# visualize
from IPython.display import Video
vwrite('vis.mp4', imgs)
Video('vis.mp4', embed=True, width=256, height=256)

Eval PushTStateEnv: 201it [00:12, 16.37it/s, reward=0.84]                           

Score:  0.9941398371494766
